# 🌆 UVIP AI - Training Pipeline (Kaggle Notebook)

**Notebook ini untuk training model UVIP AI di Kaggle (GRATIS 30 jam GPU/bulan)**

## Workflow:
1. Setup environment & install dependencies
2. Upload data (434 photos + labels)
3. Extract features (SegFormer + DINOv2)
4. Train XGBoost models (Beauty, Safety, Comfort, UVI)
5. Download trained models

**Requirements**:
- Enable GPU: Settings → Accelerator → GPU T4 x2
- Internet access: Settings → Internet → On

**Estimated time**: ~2-3 jam
**Cost**: $0 (Kaggle free tier)

## Step 1: Setup Environment

In [ ]:
# Check GPU
!nvidia-smi

# Check Python version
import sys
print(f"Python version: {sys.version}")

In [ ]:
# Install dependencies
!pip install -q ultralytics transformers timm xgboost shap opencv-python-headless
!pip install -q scikit-learn pandas tqdm pillow pydantic-settings python-dotenv

# Verify installation
import torch
print(f"PyTorch version: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.2f} GB")

## Step 2: Clone Repository & Setup

In [ ]:
# Clone UVIP AI repository
# Ganti dengan repo URL Anda jika sudah di-push ke GitHub
# !git clone https://github.com/your-username/uvip-ai.git

# Untuk sekarang, kita buat struktur manual
import os
os.makedirs('uvip-ai/src/uvip_ai', exist_ok=True)
os.makedirs('uvip-ai/data/extracted/photos', exist_ok=True)
os.makedirs('uvip-ai/data/training', exist_ok=True)
os.makedirs('uvip-ai/models/perception', exist_ok=True)

print("✓ Repository structure created")

## Step 3: Upload Data

**Upload 2 file ini ke Kaggle**:
1. `photos.zip` - 434 foto dari `data/extracted/photos/`
2. `labels.csv` - Survey labels yang sudah diisi

Cara upload:
- Klik tombol "Add Input" di sidebar kanan
- Upload files atau create dataset

In [ ]:
# Load existing extracted photos
from pathlib import Path

input_dir = Path('/kaggle/input')
output_dir = Path('/kaggle/working/uvip-ai/data/extracted/photos')

output_dir.mkdir(parents=True, exist_ok=True)

# Cari semua file gambar di dataset Kaggle
image_extensions = {'.jpg', '.jpeg', '.png', '.webp', '.bmp'}

image_files = [
    p for p in input_dir.rglob('*')
    if p.is_file() and p.suffix.lower() in image_extensions
]

print(f"✅ Ditemukan {len(image_files)} file gambar.")

# Tampilkan beberapa contoh
for img in image_files[:10]:
    print(img)

In [ ]:
# Load labels
import pandas as pd

# Ganti dengan path labels.csv Anda
# labels_path = '/kaggle/input/uvip-data/labels.csv'
labels_path = '/kaggle/input/uvip-data/labels.csv'

if Path(labels_path).exists():
    labels_df = pd.read_csv(labels_path)
    print(f"✓ Labels loaded: {len(labels_df)} rows")
    print(f"\nColumns: {list(labels_df.columns)}")
    print(f"\nSample:")
    print(labels_df.head())
else:
    print("⚠️ labels.csv not found. Please upload via 'Add Input' button.")

## Step 4: Feature Extraction

Extract features dari semua foto:
- **SegFormer-B0**: 5 metrik urban (2GB VRAM)
- **DINOv2-Base**: 768-d embedding (3GB VRAM)

**Total**: ~5GB VRAM (cukup untuk T4 16GB)

In [ ]:
import cv2
import numpy as np
import torch
from PIL import Image
from tqdm import tqdm
from transformers import (
    SegformerForSemanticSegmentation,
    SegformerImageProcessor,
    AutoImageProcessor,
    Dinov2Model
)
import torch.nn.functional as F

# Load SegFormer-B0 (lebih kecil, 2GB VRAM)
print("Loading SegFormer-B0...")
seg_processor = SegformerImageProcessor.from_pretrained('nvidia/segformer-b0-finetuned-ade-512-512')
seg_model = SegformerForSemanticSegmentation.from_pretrained(
    'nvidia/segformer-b0-finetuned-ade-512-512'
).to('cuda').to(torch.float16).eval()

# Load DINOv2-Base (768-d, 3GB VRAM)
print("Loading DINOv2-Base...")
dinov2_processor = AutoImageProcessor.from_pretrained('facebook/dinov2-base')
dinov2_model = Dinov2Model.from_pretrained(
    'facebook/dinov2-base'
).to('cuda').to(torch.float16).eval()

print("✓ Models loaded")

In [ ]:
def extract_segmentation_metrics(image_path):
    """Extract 5 urban metrics dari foto."""
    img = Image.open(image_path).convert('RGB')
    inputs = seg_processor(images=img, return_tensors='pt').to('cuda')
    
    with torch.no_grad():
        outputs = seg_model(**inputs)
        logits = outputs.logits.to(torch.float32)
        preds = logits.argmax(dim=1).squeeze().cpu().numpy()
    
    # Resize to original size
    orig_w, orig_h = img.size
    preds_resized = cv2.resize(preds, (orig_w, orig_h), interpolation=cv2.INTER_NEAREST)
    
    # Calculate metrics (simplified)
    total_pixels = preds_resized.size
    class_counts = np.bincount(preds_resized.flatten(), minlength=19)
    pct = class_counts / total_pixels * 100
    
    # ADE20K class mapping (simplified)
    # 0: wall, 1: building, 2: sky, 3: floor, 4: tree, 5: ceiling, 6: road
    # 7: bed, 8: window, 9: grass, 10: cabinet, 11: sidewalk, 12: person
    # 13: earth, 14: door, 15: table, 16: mountain, 17: plant, 18: curtain
    
    vegetation_pct = (pct[4] + pct[9] + pct[17]) if len(pct) > 17 else 0  # tree + grass + plant
    building_pct = (pct[0] + pct[1]) if len(pct) > 1 else 0  # wall + building
    sky_pct = pct[2] if len(pct) > 2 else 0
    road_pct = pct[6] if len(pct) > 6 else 0
    sidewalk_pct = pct[11] if len(pct) > 11 else 0
    
    green_coverage = vegetation_pct + sky_pct
    walkability = sidewalk_pct / (sidewalk_pct + road_pct + 1e-6)
    visual_clutter = building_pct / (green_coverage + 1)
    
    return {
        'green_coverage_pct': round(green_coverage, 4),
        'building_coverage_pct': round(building_pct, 4),
        'walkability_ratio': round(walkability, 4),
        'visual_clutter_index': round(visual_clutter, 4),
        'sky_visibility_pct': round(sky_pct, 4)
    }

@torch.inference_mode()
def extract_dinov2_embedding(image_path):
    """Extract 768-d embedding dari foto."""
    img = Image.open(image_path).convert('RGB')
    inputs = dinov2_processor(images=img, return_tensors='pt').to('cuda')
    
    with torch.autocast(device_type='cuda', dtype=torch.float16):
        outputs = dinov2_model(**inputs)
    
    # CLS token embedding
    embedding = outputs.last_hidden_state[:, 0, :].squeeze().cpu().numpy()
    embedding = embedding / np.linalg.norm(embedding)  # L2 normalize
    
    return embedding.astype(np.float32)

print("✓ Feature extraction functions defined")

In [ ]:
# ==========================================
# EXTRACT FEATURES (Self-Contained)
# ==========================================

import cv2
import numpy as np
import torch
from PIL import Image
from tqdm import tqdm
from transformers import (
    SegformerForSemanticSegmentation,
    SegformerImageProcessor,
    AutoImageProcessor,
    Dinov2Model
)
import torch.nn.functional as F

# ==========================================
# LOAD MODELS (if not already loaded)
# ==========================================

if 'seg_model' not in globals():
    print("Loading SegFormer-B0...")
    seg_processor = SegformerImageProcessor.from_pretrained('nvidia/segformer-b0-finetuned-ade-512-512')
    seg_model = SegformerForSemanticSegmentation.from_pretrained(
        'nvidia/segformer-b0-finetuned-ade-512-512'
    ).to('cuda').to(torch.float16).eval()
    print("✓ SegFormer loaded")

if 'dinov2_model' not in globals():
    print("Loading DINOv2-Base...")
    dinov2_processor = AutoImageProcessor.from_pretrained('facebook/dinov2-base')
    dinov2_model = Dinov2Model.from_pretrained(
        'facebook/dinov2-base'
    ).to('cuda').to(torch.float16).eval()
    print("✓ DINOv2 loaded")

# ==========================================
# FEATURE EXTRACTION FUNCTIONS
# ==========================================

def extract_segmentation_metrics(image_path):
    """Extract segmentation metrics dari gambar."""
    img = Image.open(image_path).convert('RGB')
    
    # SegFormer inference
    inputs = seg_processor(images=img, return_tensors="pt").to('cuda')
    inputs['pixel_values'] = inputs['pixel_values'].to(torch.float16)  # FIX: cast to float16
    
    with torch.no_grad():
        outputs = seg_model(**inputs)
    
    # Get segmentation map
    seg_map = outputs.logits.argmax(dim=1)[0].cpu().numpy()
    
    # Metrics sederhana
    unique_classes = len(np.unique(seg_map))
    main_class_ratio = np.max(np.bincount(seg_map.flatten())) / seg_map.size
    
    return {
        'num_classes': unique_classes,
        'main_class_ratio': float(main_class_ratio),
        'complexity': float(unique_classes / 150),  # ADE20K has 150 classes
        'uniformity': float(1 - main_class_ratio),
        'diversity': float(unique_classes / 50),  # Normalized
    }

def extract_dinov2_embedding(image_path):
    """Extract DINOv2 embedding (768-d)."""
    img = Image.open(image_path).convert('RGB')
    
    inputs = dinov2_processor(images=img, return_tensors="pt").to('cuda')
    inputs['pixel_values'] = inputs['pixel_values'].to(torch.float16)  # FIX: cast to float16
    
    with torch.no_grad():
        outputs = dinov2_model(**inputs)
    
    # Use CLS token embedding
    embedding = outputs.last_hidden_state[:, 0, :].cpu().numpy()[0]
    return embedding

# ==========================================
# LOAD MANIFEST + RESOLVE PHOTO PATHS
# ==========================================

from pathlib import Path
import pandas as pd

DATASET_ROOT = Path('/kaggle/input/datasets/adryanprawira/dataset/extracted')
PHOTOS_DIR = DATASET_ROOT / 'photos'
MANIFEST_PATH = DATASET_ROOT / 'manifest.csv'

# Check files
if not MANIFEST_PATH.exists():
    raise FileNotFoundError(f"manifest.csv tidak ditemukan: {MANIFEST_PATH}")

if not PHOTOS_DIR.exists():
    raise FileNotFoundError(f"Folder photos tidak ditemukan: {PHOTOS_DIR}")

# Load manifest
manifest_df = pd.read_csv(MANIFEST_PATH)

print("=" * 60)
print("UVIP-AI DATASET")
print("=" * 60)

print(f"Manifest : {MANIFEST_PATH}")
print(f"Photos   : {PHOTOS_DIR}")
print(f"Records  : {len(manifest_df)}")

# Resolve photo paths
manifest_df['photo_path'] = (
    PHOTOS_DIR.parent / manifest_df['relative_path']
).apply(Path)

# Check existence
manifest_df['exists'] = manifest_df['photo_path'].apply(lambda p: p.exists())

found = manifest_df['exists'].sum()
missing = (~manifest_df['exists']).sum()

print(f"\n✅ Photos found   : {found}")
print(f"❌ Photos missing : {missing}")

# Only use valid photos
valid_manifest = manifest_df[manifest_df['exists']].copy()
photo_files = valid_manifest['photo_path'].tolist()

print(f"\n✅ Ready for processing: {len(photo_files)} photos")

# ==========================================
# EXTRACT FEATURES
# ==========================================

features_list = []
for photo_path in tqdm(photo_files, desc="Extracting features"):
    try:
        # Segmentation metrics
        seg_metrics = extract_segmentation_metrics(photo_path)
        
        # DINOv2 embedding
        embedding = extract_dinov2_embedding(photo_path)
        
        # Build feature row
        row = {
            'filename': photo_path.name,
            'area': photo_path.parent.name,
            'point_id': photo_path.stem,
        }
        row.update(seg_metrics)
        
        # Add embedding (768-d)
        for i, val in enumerate(embedding):
            row[f'emb_{i}'] = val
        
        features_list.append(row)
    except Exception as e:
        print(f"Error processing {photo_path}: {e}")
        continue

# Save to CSV
features_output = Path('/kaggle/working/uvip-ai/data/training/features.csv')
features_output.parent.mkdir(parents=True, exist_ok=True)

features_df = pd.DataFrame(features_list)
features_df.to_csv(features_output, index=False)

print(f"\n✓ Features extracted: {len(features_df)} photos")
print(f"✓ Saved to: {features_output}")
print(f"✓ Columns: {len(features_df.columns)} (5 seg metrics + 768 embedding)")

if len(features_df) == 0:
    raise ValueError("❌ Tidak ada features yang berhasil di-extract!")

In [ ]:
# Free GPU memory
del seg_model, dinov2_model
torch.cuda.empty_cache()
print("✓ GPU memory freed")

## Step 5: Merge Features with Labels

In [ ]:
# ==========================================
# LOAD FEATURES
# ==========================================

import pandas as pd
from pathlib import Path

features_path = Path('/kaggle/working/uvip-ai/data/training/features.csv')

# Check if file exists and has content
if not features_path.exists():
    raise FileNotFoundError(f"❌ File tidak ditemukan: {features_path}\n   Pastikan cell 12 (extract features) sudah dijalankan!")

file_size = features_path.stat().st_size
if file_size == 0:
    raise ValueError(f"❌ File kosong: {features_path}\n   Re-run cell 12!")

# Load features
features_df = pd.read_csv(features_path)

if features_df.empty:
    raise ValueError(f"❌ DataFrame kosong setelah load dari {features_path}\n   Re-run cell 12!")

print(f"✓ Features loaded: {features_df.shape}")
print(f"✓ Columns: {len(features_df.columns)} (filename, area, point_id, 5 seg metrics, 768 embeddings)")
print(f"\n📊 Sample data:")
print(features_df[['filename', 'area', 'point_id']].head())
print(f"\n✓ features_df ready for merge")

In [ ]:
# ==========================================
# LOAD LABELS (with synthetic fallback)
# ==========================================

import cv2
import numpy as np
from tqdm import tqdm
import warnings
warnings.filterwarnings('ignore')

# Dataset root (same as cell 12)
DATASET_ROOT = Path('/kaggle/input/datasets/adryanprawira/dataset/extracted')
PHOTOS_DIR = DATASET_ROOT / 'photos'
MANIFEST_PATH = DATASET_ROOT / 'manifest.csv'

# Try multiple label sources
labels_candidates = [
    DATASET_ROOT / 'labels.csv',
    DATASET_ROOT / 'labels_synthetic.csv',
    Path('/kaggle/working/uvip-ai/data/training/labels.csv'),
    Path('/kaggle/working/uvip-ai/data/training/labels_synthetic.csv'),
]

labels_path = None
for candidate in labels_candidates:
    if candidate.exists():
        labels_path = candidate
        break

if labels_path:
    print(f"✓ Loading labels from: {labels_path}")
    labels_df = pd.read_csv(labels_path)
    print(f"✓ Labels loaded: {labels_df.shape}")
else:
    print("⚠️  labels.csv tidak ditemukan. Generating synthetic labels dari images + manifest...")
    
    if not MANIFEST_PATH.exists():
        raise FileNotFoundError(f"❌ Manifest tidak ditemukan: {MANIFEST_PATH}")
    
    print(f"✓ Using manifest: {MANIFEST_PATH}")
    manifest_df = pd.read_csv(MANIFEST_PATH)
    print(f"✓ Manifest loaded: {len(manifest_df)} photos")
    
    # Resolve photo paths
    manifest_df['photo_path'] = (
        PHOTOS_DIR.parent / manifest_df['relative_path']
    ).apply(Path)
    
    # Filter existing photos
    manifest_df['exists'] = manifest_df['photo_path'].apply(lambda p: p.exists())
    valid_manifest = manifest_df[manifest_df['exists']].copy()
    
    # Generate synthetic labels from images
    def extract_image_features_for_labels(image_path):
        img = cv2.imread(str(image_path))
        if img is None:
            return None
        img_hsv = cv2.cvtColor(img, cv2.COLOR_BGR2HSV)
        img_gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)
        
        brightness = np.mean(img_gray) / 255.0
        saturation = np.mean(img_hsv[:, :, 1]) / 255.0
        
        # Green detection (vegetation proxy)
        lower_green = np.array([35, 50, 50])
        upper_green = np.array([85, 255, 255])
        green_mask = cv2.inRange(img_hsv, lower_green, upper_green)
        green_ratio = np.sum(green_mask > 0) / (img.shape[0] * img.shape[1])
        
        # Color diversity
        img_rgb = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
        small_img = cv2.resize(img_rgb, (50, 50))
        unique_colors = len(np.unique(small_img.reshape(-1, 3), axis=0))
        color_diversity = min(unique_colors / 100, 1.0)
        
        # Edge density
        edges = cv2.Canny(img_gray, 50, 150)
        edge_density = np.mean(edges > 0)
        
        return brightness, saturation, green_ratio, color_diversity, edge_density
    
    # Process all images
    labels_list = []
    
    print("\n🔍 Generating synthetic labels dari images...")
    for _, row in tqdm(valid_manifest.iterrows(), total=len(valid_manifest), desc="Processing"):
        photo_path = row['photo_path']
        filename = row['filename']
        
        features = extract_image_features_for_labels(photo_path)
        if features is None:
            continue
        
        brightness, saturation, green_ratio, color_diversity, edge_density = features
        
        # Compute synthetic scores (1-10)
        beauty = 1 + (saturation * 0.4 + color_diversity * 0.4 + edge_density * 0.2) * 9
        safety = 1 + brightness * 9
        comfort = 1 + (green_ratio * 0.6 + (1 - abs(brightness - 0.5) * 2) * 0.4) * 9
        uvi = 1 + (green_ratio * 0.7 + saturation * 0.3) * 9
        
        labels_list.append({
            'filename': filename,
            'label_beauty': round(beauty, 2),
            'label_safety': round(safety, 2),
            'label_comfort': round(comfort, 2),
            'label_uvi': round(uvi, 2),
        })
    
    if not labels_list:
        raise ValueError("❌ Tidak ada gambar yang bisa diproses untuk synthetic labels!")
    
    labels_df = pd.DataFrame(labels_list)
    
    # Save synthetic labels
    synthetic_path = Path('/kaggle/working/uvip-ai/data/training/labels_synthetic.csv')
    synthetic_path.parent.mkdir(parents=True, exist_ok=True)
    labels_df.to_csv(synthetic_path, index=False)
    
    print(f"\n✓ Synthetic labels generated: {len(labels_df)} photos")
    print(f"✓ Saved to: {synthetic_path}")
    print(f"\n📊 Label statistics:")
    for col in ['label_beauty', 'label_safety', 'label_comfort', 'label_uvi']:
        print(f"   {col:15s}: mean={labels_df[col].mean():.2f}, "
              f"std={labels_df[col].std():.2f}, "
              f"min={labels_df[col].min():.2f}, max={labels_df[col].max():.2f}")
    print(f"\n⚠️  Labels ini SYNTHETIC (dari image features, bukan survey manusia)")
    print(f"   Cocok untuk testing pipeline. Untuk production, pakai data survey asli.")

print(f"\n✓ labels_df ready: {labels_df.shape}")
print(f"✓ Columns: {list(labels_df.columns)}")
print(f"\n📊 Sample:")
print(labels_df.head())

In [ ]:
# ==========================================
# MERGE FEATURES WITH LABELS
# ==========================================

import pandas as pd

# Merge features with labels
dataset_df = pd.merge(features_df, labels_df, on='filename', how='inner')
print(f"\n✓ Merged dataset: {dataset_df.shape}")

# Check for missing labels
label_cols = ['label_beauty', 'label_safety', 'label_comfort', 'label_uvi']
missing = dataset_df[label_cols].isnull().sum()
print(f"\nMissing labels per column:")
print(missing)

# Drop rows with missing labels
dataset_df = dataset_df.dropna(subset=label_cols)
print(f"\n✓ Final dataset after dropping NaN: {len(dataset_df)} rows")

# Save
dataset_output = '/kaggle/working/uvip-ai/data/training/dataset.csv'
dataset_df.to_csv(dataset_output, index=False)
print(f"✓ Saved to: {dataset_output}")

print(f"\n📊 Dataset summary:")
print(f"   Total rows: {len(dataset_df)}")
print(f"   Total columns: {len(dataset_df.columns)}")
print(f"   Areas: {dataset_df['area'].nunique()}")
print(f"   Label columns: {label_cols}")

In [ ]:
# ==========================================
# PREPARE TRAINING DATA
# ==========================================

import xgboost as xgb
from sklearn.model_selection import KFold
from sklearn.metrics import r2_score, mean_absolute_error, mean_squared_error
import json

# Load dataset
dataset_path = '/kaggle/working/uvip-ai/data/training/dataset.csv'
dataset_df = pd.read_csv(dataset_path)

print(f"✓ Dataset loaded: {dataset_df.shape}")

# Feature columns
seg_cols = ['num_classes', 'main_class_ratio', 'complexity', 'uniformity', 'diversity']
emb_cols = [c for c in dataset_df.columns if c.startswith('emb_')]
feature_cols = seg_cols + emb_cols

print(f"✓ Features: {len(feature_cols)} (5 seg metrics + {len(emb_cols)} embedding)")

# Targets
targets = ['label_beauty', 'label_safety', 'label_comfort', 'label_uvi']
target_names = ['beauty', 'safety', 'comfort', 'uvi']

print(f"✓ Targets: {targets}")
print(f"\n📊 Dataset ready for training")

In [ ]:
def train_xgboost_model(X, y, n_folds=5):
    """Train XGBoost dengan K-Fold cross-validation."""
    kf = KFold(n_splits=n_folds, shuffle=True, random_state=42)
    
    r2_scores = []
    mae_scores = []
    rmse_scores = []
    
    for fold, (train_idx, val_idx) in enumerate(kf.split(X)):
        X_train, X_val = X[train_idx], X[val_idx]
        y_train, y_val = y[train_idx], y[val_idx]
        
        # Train model (FIX: early_stopping_rounds di constructor, bukan di fit)
        model = xgb.XGBRegressor(
            n_estimators=500,
            max_depth=6,
            learning_rate=0.05,
            subsample=0.8,
            colsample_bytree=0.8,
            objective='reg:squarederror',
            n_jobs=-1,
            random_state=42,
            tree_method='hist',
            early_stopping_rounds=50  # FIX: pindah ke constructor
        )
        
        model.fit(
            X_train, y_train,
            eval_set=[(X_val, y_val)],
            verbose=False
        )
        
        # Predict
        y_pred = model.predict(X_val)
        
        # Metrics
        r2 = r2_score(y_val, y_pred)
        mae = mean_absolute_error(y_val, y_pred)
        rmse = np.sqrt(mean_squared_error(y_val, y_pred))
        
        r2_scores.append(r2)
        mae_scores.append(mae)
        rmse_scores.append(rmse)
    
    return {
        'r2_mean': np.mean(r2_scores),
        'r2_std': np.std(r2_scores),
        'mae_mean': np.mean(mae_scores),
        'rmse_mean': np.mean(rmse_scores),
        'model': model  # Return last model
    }

print("✓ Training function defined")

In [ ]:
# Train models untuk semua target
import pickle
import os

# Create output directory
os.makedirs('uvip-ai/models/perception', exist_ok=True)

# Use correct feature columns (from cell 12)
seg_cols = ['num_classes', 'main_class_ratio', 'complexity', 'uniformity', 'diversity']
emb_cols = [c for c in dataset_df.columns if c.startswith('emb_')]
feature_cols = seg_cols + emb_cols

print(f"✓ Using {len(feature_cols)} features: {len(seg_cols)} seg metrics + {len(emb_cols)} embeddings")

X = dataset_df[feature_cols].fillna(0).values
results = {}

for target, name in zip(targets, target_names):
    print(f"\n{'='*60}")
    print(f"Training {name.upper()} model...")
    print(f"{'='*60}")
    
    y = dataset_df[target].values
    
    # Train
    result = train_xgboost_model(X, y, n_folds=5)
    
    # Print metrics
    print(f"R²: {result['r2_mean']:.4f} ± {result['r2_std']:.4f}")
    print(f"MAE: {result['mae_mean']:.4f}")
    print(f"RMSE: {result['rmse_mean']:.4f}")
    
    # Check if target achieved
    if result['r2_mean'] >= 0.7:
        print(f"✓ PASS (R² ≥ 0.7)")
    else:
        print(f"✗ FAIL (R² < 0.7)")
    
    # Save model
    model_path = f'uvip-ai/models/perception/{name}_xgb.pkl'
    with open(model_path, 'wb') as f:
        pickle.dump(result['model'], f)
    print(f"✓ Model saved: {model_path}")
    
    # Store results
    results[name] = {
        'r2_mean': result['r2_mean'],
        'r2_std': result['r2_std'],
        'mae_mean': result['mae_mean'],
        'rmse_mean': result['rmse_mean']
    }

# Save metrics
with open('uvip-ai/models/perception/metrics.json', 'w') as f:
    json.dump(results, f, indent=2)

print(f"\n{'='*60}")
print("TRAINING COMPLETE")
print(f"{'='*60}")

## Step 7: Download Trained Models

In [ ]:
# List trained models
import os

models_dir = 'uvip-ai/models/perception'
print("Trained models:")
for file in os.listdir(models_dir):
    file_path = os.path.join(models_dir, file)
    size = os.path.getsize(file_path) / 1024
    print(f"  {file} ({size:.1f} KB)")

In [ ]:
# Create zip for download
import zipfile

models_dir = 'uvip-ai/models/perception'

# Create zip file
with zipfile.ZipFile('trained_models.zip', 'w') as zipf:
    for file in os.listdir(models_dir):
        file_path = os.path.join(models_dir, file)
        zipf.write(file_path, arcname=file)

print("✓ Models zipped: trained_models.zip")
print(f"✓ Size: {os.path.getsize('trained_models.zip') / 1024:.1f} KB")
print("\n📥 Download via Kaggle sidebar → Output → 'trained_models.zip' → click download icon")
print("   Or use: from kaggle_api_client import download (if available)")

## Step 8: Summary & Next Steps

In [ ]:
# Download trained models from Kaggle Output
# In Kaggle, files in /kaggle/working/ appear in the Output panel on the right sidebar.
# Click the download icon next to 'trained_models.zip' to download it.

print("✓ Training complete!")
print("\n📥 To download trained_models.zip:")
print("   1. Look at the right sidebar → 'Output' section")
print("   2. Find 'trained_models.zip'")
print("   3. Click the download icon (↓)")
print("\nOr copy to your Kaggle output directory:")
import shutil
if os.path.exists('trained_models.zip'):
    shutil.copy('trained_models.zip', '/kaggle/working/trained_models.zip')
    print("✓ Copied to /kaggle/working/trained_models.zip")